## Tool Schemas Claude Selects Correctly: Definition, Loop, and Calling Patterns


- Claude reads the tool definitions,
- Decides which one fits the situations
- Tell your app what to call it along with the required input.


#### Proper Implementation:
1. **Define Schema**:
    - name
    - Description
    - Input Schema

2. **Send Message** : your code sends a message to claude including the tool definition adn the users input

3. **tool_use Block** :
    - Claude issues a tool_use block with:
        - tool name
        - a Unique ID
        - The input arguments
    The API response comes back with 'stop_reason : tool_use'

4. **Execute tool** : Your code executes the tool using those arguments.
    - Note that the assistant turn has already ended (Claude is not holding a connection open or waiting on your server).
    - The model is stateless between calls.
    - To continue, your code makes a fresh API request containing the prior messages plus the tool result.

5. **Return result** : You return the result in a 'tool_result' block that references the original 'tool_use' ID.

6. **Claude Continues** : Claude continues using the tool result as context for its next response, either another toll-block     or a final end turn


### Message block structure in a tool-use conversation

- Structured Blocks
- Assistant turn and User turn
- 4 block types for tool_use session
    - text_block : Claude's prose response
    - tool_use block : tool call (tool name, unique ID and input arguments)
    - tool_result block : code return after running the tool
    - thinking block : internal reasoning (when extended thinking is enabled)

Note: Every tool_use block in an assistant turn must be answered by tool_result block with a matching ID in the user turn that immediately follows.
- If:
    - IDs dont match
    - result is missing
    - turns are out of order

The request fails validation (Prompt adjustment won't work)

**_Text Block_**
- _Role_: Assistant/Claude
- _Contains_: Claude’s prose output
- _Critical Rule_ :
-     - Claude can return a text block with a tool_use block in the same turn.
      - your code must preserve the full content array (along with text block,when appending that turn to
      - conversation history. otherwise it corrupts the context

**_tool_use block_**
- _Role_ : Assistant/Claude
- _Contains_ : tool name, unique ID, input argument
- _Critical Rule_ : Every tool_use block must be answered by a tool_result block in the immediately following user turn with same ID

**_tool_result Block_**
- _Role_ : User
- _Contains_ :
        - Matching tool_use ID,
        - the result content and
        - is_error flag (true)
- _Critical Rule_ :
-     - The tool_use_id value must match the original tool_use block exactly.
      - Claude uses this ID to connect each result back to the call that produced it,
      - which matters when a single assistant turn issues multiple tool calls and the results arrive in a different order.

**_thinking Block_**
- _Role_ : Assistant (extended thinking only)/Claude
- _Contains_ : Claude's internal reasoning, only visible when enabled
- _Critical Rule_ : The block must be passed back to the api unchanged in subsequent turns.

### Schema anatomy: What Claude reads to make a tool selection decision
1. **Name**: A short identifier that should be specific. For example, get_account_balance is more useful to Claude than get_data.
2. **Description**: A critical part that Claude reads to decide whether a tool is required or not. You should always write the description in two parts, including when to and when not to use the tool:
    - A description that says "use this to find information" will cause wrong selections because Claude cannot distinguish it from any other tool that retrieves something.
    - A description that says "use this to retrieve the current balance for a specific account ID and do not use this for transaction history" gives Claude an exclusion condition to work with and is appropriately descriptive.
3. **input_schema**: Defines the parameters (the inputs your tool function accepts) using JSON Schema.
    - You should mark parameters as required when Claude requires them to call the tool correctly.
    - You can mark parameters as optional when the tool can operate without them. Overlapping parameter types between tools is the most common source of wrong-tool calls.

### Decision table: Schema design choices